## GenAI Disclosure Statement

Generative AI tools were used as learning aids. All analysis and conclusions are the team's own work.

---

# Notebook 06 — Security, Abuse Pathways, and Attack-Surface Assessment
### DNSC 6330 Responsible Machine Learning Capstone | GWU

**Purpose:** Document the threat model, access-control specification, and adversarial failure modes
for a deployed HMDA scoring system. Produces the security component of the audit record.

**Inputs:** System card, threat actor framework  
**Outputs:** `tables/threat_model_table.csv`, `tables/access_control_matrix.csv`,
narrative security assessment in `docs/06_security_residual_risk.md`

**Lecture 05 connection:** ML security, abuse pathways, poisoning resistance, access controls.


In [ ]:
import pandas as pd
import os

TABLES_DIR = os.path.join(os.getcwd(), "..", "tables")
os.makedirs(TABLES_DIR, exist_ok=True)


## Section 6.1 — Threat Actor Framework

Deployed HMDA scoring models operate in an adversarial environment. Unlike generic ML systems,
an HMDA-linked model sits at the intersection of financial incentives, consumer rights,
and regulatory scrutiny — creating a distinct threat landscape.

See `docs/06_security_residual_risk.md` for the full narrative.


In [ ]:
threat_actors = pd.DataFrame([
    {
        "Threat Actor": "Mortgage brokers",
        "Motivation": "Maximize client approval rates for commission income",
        "Capability": "High domain knowledge; repeated system access",
        "Entry Point": "Application fields before submission",
        "Likelihood": "High",
    },
    {
        "Threat Actor": "Loan officers",
        "Motivation": "Approval quota pressure; avoid underwriting liability",
        "Capability": "Operational access to system inputs",
        "Entry Point": "Manual input fields; system overrides",
        "Likelihood": "Medium",
    },
    {
        "Threat Actor": "Third-party data vendors",
        "Motivation": "Contract revenue; indifference to data quality",
        "Capability": "Access to training data inputs (census, ACS features)",
        "Entry Point": "Training data pipeline",
        "Likelihood": "Low–Medium",
    },
    {
        "Threat Actor": "Internal adversaries",
        "Motivation": "Cover discriminatory practices; manipulate model",
        "Capability": "Insider access",
        "Entry Point": "Training data; model configuration; label pipeline",
        "Likelihood": "Low",
    },
    {
        "Threat Actor": "External attackers",
        "Motivation": "Model extraction; competitive intelligence",
        "Capability": "Medium; requires repeated API access",
        "Entry Point": "Scoring API / portal",
        "Likelihood": "Low–Medium",
    },
])
display(threat_actors)


## Section 6.2 — Threat Scenario Table

In [ ]:
threat_table = pd.DataFrame([
    {
        "ID": "T-01",
        "Scenario": "Income/DTI inflation by broker",
        "Actor": "Mortgage broker",
        "Entry Point": "income, debt_to_income_ratio fields at application entry",
        "Impact": "Inflated approvals; increased credit risk; model gaming normalizes",
        "Likelihood": "High",
        "Mitigation": "Income plausibility checks; outlier flagging on broker channels; third-party verification",
        "Residual Risk": "Medium",
    },
    {
        "ID": "T-02",
        "Scenario": "Poisoning via vendor data corruption of census-tract features",
        "Actor": "Third-party data vendor",
        "Entry Point": "Training data pipeline (ACS / census features)",
        "Impact": "Proxy-risk amplification; disparate outcomes shift without triggering performance alerts",
        "Likelihood": "Medium",
        "Mitigation": "SHA-256 hash validation; PSI check before retraining; human review if PSI > 0.10 on proxy features",
        "Residual Risk": "Low–Medium",
    },
    {
        "ID": "T-03",
        "Scenario": "Scoring API extraction to reverse-engineer decision boundary",
        "Actor": "External attacker",
        "Entry Point": "Production scoring API / portal",
        "Impact": "Model IP exfiltration; systematic gaming guide created",
        "Likelihood": "Low–Medium",
        "Mitigation": "Rate-limiting; anomaly detection on query volume; return binary decision not raw score",
        "Residual Risk": "Low",
    },
    {
        "ID": "T-04",
        "Scenario": "Label manipulation by insider to suppress minority approval rates in training data",
        "Actor": "Internal adversary",
        "Entry Point": "Training data or label construction pipeline",
        "Impact": "Discriminatory model baked in silently; may be invisible at model-level monitoring",
        "Likelihood": "Low",
        "Mitigation": "Audit trail on label modifications; 4-eyes principle for training data changes",
        "Residual Risk": "Low",
    },
    {
        "ID": "T-05",
        "Scenario": "Broker consortium boundary probing to map decision surface for minority applicants",
        "Actor": "Broker consortium",
        "Entry Point": "Repeated scoring via application portal",
        "Impact": "Targeted gaming of decision boundary near AIR = 0.80",
        "Likelihood": "Medium",
        "Mitigation": "Score-smoothing; threshold obfuscation; broker-channel monitoring",
        "Residual Risk": "Medium",
    },
])
display(threat_table)
threat_table.to_csv(os.path.join(TABLES_DIR, "threat_model_table.csv"), index=False)
print("Threat model table saved.")


## Section 6.3 — Access-Control Matrix

In [ ]:
access_control = pd.DataFrame([
    {
        "Role": "Data Engineer",
        "Train Model": " (pipeline only)",
        "Score Application": "✗",
        "View Raw Features": "",
        "View Prediction Score": "✗",
        "View Protected Attributes": "Masked",
        "View Model Weights": "✗",
        "Modify Training Data": " (with audit trail)",
    },
    {
        "Role": "ML Engineer",
        "Train Model": "",
        "Score Application": " (test/staging)",
        "View Raw Features": "",
        "View Prediction Score": "",
        "View Protected Attributes": "Masked",
        "View Model Weights": " (read only)",
        "Modify Training Data": " (with 4-eyes)",
    },
    {
        "Role": "Underwriter",
        "Train Model": "✗",
        "Score Application": "View output only",
        "View Raw Features": "✗",
        "View Prediction Score": "Binary only",
        "View Protected Attributes": "✗",
        "View Model Weights": "✗",
        "Modify Training Data": "✗",
    },
    {
        "Role": "Compliance Officer",
        "Train Model": "✗",
        "Score Application": "✗",
        "View Raw Features": "Aggregated reports",
        "View Prediction Score": " (aggregate)",
        "View Protected Attributes": " (aggregate)",
        "View Model Weights": "✗",
        "Modify Training Data": "✗",
    },
    {
        "Role": "Model Risk Officer",
        "Train Model": "✗ (audit only)",
        "Score Application": " (validation testing)",
        "View Raw Features": "",
        "View Prediction Score": "",
        "View Protected Attributes": " (for validation)",
        "View Model Weights": " (read only)",
        "Modify Training Data": "✗",
    },
    {
        "Role": "External Auditor",
        "Train Model": "✗",
        "Score Application": "✗",
        "View Raw Features": "Aggregated only",
        "View Prediction Score": "Aggregated only",
        "View Protected Attributes": " (aggregate)",
        "View Model Weights": "✗",
        "Modify Training Data": "✗",
    },
    {
        "Role": "Applicant",
        "Train Model": "✗",
        "Score Application": "✗",
        "View Raw Features": "Own record only",
        "View Prediction Score": "Denial reason (legally required)",
        "View Protected Attributes": "Own record only",
        "View Model Weights": "✗",
        "Modify Training Data": "✗",
    },
])
display(access_control)
access_control.to_csv(os.path.join(TABLES_DIR, "access_control_matrix.csv"), index=False)
print("Access control matrix saved.")


## Section 6.4 — Adversarial Failure Modes

The three most plausible adversarial failure modes for a deployed HMDA scoring system:

**1. Silent gaming normalization:**  
Broker gaming of income/DTI fields becomes industry-standard practice. The model's effective
training distribution shifts over time to include inflated inputs, requiring periodic recalibration.
If gaming is not race-neutral, it may silently affect AIR in either direction.

**2. Proxy feature drift:**  
A geographic rezoning event, census tract boundary change, or neighborhood demographic shift
changes the meaning of census-tract features the model relies on. The model's predictions change
without any change in the model itself. PSI may not detect gradual shifts.

**3. Monitoring system failure:**  
Protected attribute data is not captured at scoring time (common if the scoring system and
application intake system are separate). AIR monitoring becomes impossible. The model continues
scoring without fairness oversight — the highest-priority deployment condition.

---

## Section 6.5 — Poisoning Resistance Discussion

If the model is retrained periodically, a third-party vendor supplying ACS or census-tract
features could corrupt those features. Because census-tract features carry elevated proxy risk
for race, corruption of those features during retraining could shift the model's disparate impact
characteristics without triggering performance-level alerts.

**Why PSI alone is insufficient:** PSI detects distribution shift, but a systematically biased
(internally consistent) dataset may have low PSI while encoding a new racial bias pattern.

**Required pre-retrain controls:**
1. SHA-256 file hash validation for all external data files before pipeline execution.
2. Automated PSI check comparing new training data to prior training data.
3. Mandatory human review of any PSI > 0.10 on proxy-risk features.
4. Separate pre-retrain fairness check: run AIR on new training data before model update.

See `docs/06_security_residual_risk.md` for complete documentation.


In [ ]:
print("Security assessment notebook complete.")
print("Full narrative: docs/06_security_residual_risk.md")
print("Tables saved:")
print("  tables/threat_model_table.csv")
print("  tables/access_control_matrix.csv")
